# ⏰ Séries Temporelles avec LSTM

## 🎯 Ce Que Vous Allez Apprendre

- Importer et manipuler des données de séries temporelles avec pandas
- Techniques pour gérer les valeurs manquantes dans les données de séries temporelles
- Visualisation de base des données avec matplotlib et seaborn
- Construction et entraînement d'un modèle LSTM simple pour l'analyse de séries temporelles

## 🛠️ Ce Que Vous Allez Créer

- Un dataset de séries temporelles nettoyé et prétraité
- Des visualisations des données de séries temporelles
- Un modèle LSTM simple pour analyser et prédire les données de séries temporelles

In [ ]:
# Partie 1: Importation et Exploration Initiale des Données
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import requests
from io import BytesIO

# Télécharger et extraire le dataset
url = 'https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/W6D4/household_power_consumption.zip'
response = requests.get(url)
with zipfile.ZipFile(BytesIO(response.content)) as z:
  z.extractall()

# Charger les données
df = pd.read_csv('household_power_consumption.txt', sep=';',
                 parse_dates={'datetime': ['Date', 'Time']},
                 infer_datetime_format=True,
                 low_memory=False,
                 na_values=['?'])

# Afficher les premières lignes
print("Premières lignes du dataset:")
print(df.head())

# Vérifier les types de données et la forme
print(f"\nForme du dataset: {df.shape}")
print(f"\nTypes de données:\n{df.dtypes}")

# Partie 2: Gestion des Valeurs Manquantes
print("\n--- Partie 2: Gestion des Valeurs Manquantes ---")

# Identifier les colonnes avec des valeurs manquantes
print("\nNombre de valeurs manquantes par colonne:")
print(df.isnull().sum())

# Remplir les valeurs manquantes avec la moyenne
for col in df.columns:
  if df[col].dtype in ['float64', 'int64']:
    df[col].fillna(df[col].mean(), inplace=True)

# Vérifier qu'il n'y a plus de valeurs manquantes
print("\nAprès traitement, valeurs manquantes:")
print(df.isnull().sum())

# Partie 3: Visualisation des Données
print("\n--- Partie 3: Visualisation des Données ---")

# Définir datetime comme index
df.set_index('datetime', inplace=True)

# Rééchantillonner et visualiser Global_active_power
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
df['Global_active_power'].resample('D').sum().plot()
plt.title('Global Active Power - Somme par Jour')
plt.xlabel('Date')
plt.ylabel('Puissance (kW)')
plt.grid(True)

plt.subplot(1, 2, 2)
df['Global_active_power'].resample('D').mean().plot(color='orange')
plt.title('Global Active Power - Moyenne par Jour')
plt.xlabel('Date')
plt.ylabel('Puissance (kW)')
plt.grid(True)

plt.tight_layout()
plt.show()

# Visualiser Global_intensity avec moyenne et écart-type
plt.figure(figsize=(14, 5))
mean_intensity = df['Global_intensity'].resample('D').mean()
std_intensity = df['Global_intensity'].resample('D').std()

plt.plot(mean_intensity.index, mean_intensity.values, label='Moyenne')
plt.fill_between(mean_intensity.index,
                  mean_intensity - std_intensity,
                  mean_intensity + std_intensity,
                  alpha=0.3, label='Écart-type')
plt.title('Global Intensity - Moyenne et Écart-type par Jour')
plt.xlabel('Date')
plt.ylabel('Intensité (A)')
plt.legend()
plt.grid(True)
plt.show()

# Partie 4: Prétraitement pour LSTM
print("\n--- Partie 4: Prétraitement pour LSTM ---")

from sklearn.preprocessing import MinMaxScaler

# Sélectionner la colonne à prédire
data = df['Global_active_power'].values.reshape(-1, 1)

# Normaliser les données
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

print(f"Forme des données normalisées: {data_scaled.shape}")

# Définir la taille de la fenêtre temporelle
time_steps = 60

# Créer les séquences pour LSTM
def create_sequences(data, time_steps):
  X, y = [], []
  for i in range(time_steps, len(data)):
    X.append(data[i-time_steps:i, 0])
    y.append(data[i, 0])
  return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, time_steps)

# Diviser en ensemble d'entraînement et de test (80/20)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Remodeler pour LSTM [samples, time_steps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"Forme X_train: {X_train.shape}")
print(f"Forme X_test: {X_test.shape}")
print(f"Forme y_train: {y_train.shape}")
print(f"Forme y_test: {y_test.shape}")

# Partie 5: Construction du Modèle LSTM
print("\n--- Partie 5: Construction du Modèle LSTM ---")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Définir l'architecture du modèle LSTM
model = keras.Sequential([
  layers.LSTM(50, return_sequences=True, input_shape=(time_steps, 1)),
  layers.Dropout(0.2),
  layers.LSTM(50, return_sequences=False),
  layers.Dropout(0.2),
  layers.Dense(25),
  layers.Dense(1)
])

# Compiler le modèle
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Afficher l'architecture
model.summary()

# Partie 6: Entraînement et Évaluation du Modèle LSTM
print("\n--- Partie 6: Entraînement et Évaluation ---")

# Entraîner le modèle
history = model.fit(
  X_train, y_train,
  validation_data=(X_test, y_test),
  epochs=10,
  batch_size=32,
  verbose=1
)

# Évaluer le modèle
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"\nPerte sur le test: {test_loss:.4f}")
print(f"MAE sur le test: {test_mae:.4f}")

# Visualiser les courbes d'apprentissage
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Perte du Modèle')
plt.xlabel('Époque')
plt.ylabel('Perte')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('MAE du Modèle')
plt.xlabel('Époque')
plt.ylabel('MAE')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Faire des prédictions
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Visualiser les prédictions
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual[:200], label='Valeurs Réelles', alpha=0.7)
plt.plot(predictions[:200], label='Prédictions', alpha=0.7)
plt.title('Prédictions LSTM vs Valeurs Réelles')
plt.xlabel('Points de Temps')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.grid(True)
plt.show()

print("\n✅ Toutes les parties sont terminées avec succès!")